# Hồi quy tuyến tính đơn: ảnh hưởng của các số đo cơ thể đến phần trăm mỡ cơ thể

**Môn:** Suy luận thống kê (MI3031) – Chương 4: Hồi quy. Tương quan
**Nhóm / Sinh viên:** *(điền tên, MSSV)*
**Dữ liệu:** Body Fat (252 nam giới)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

pd.set_option("display.precision", 3)
plt.rcParams["figure.dpi"] = 110

## 1. Giới thiệu dữ liệu

### 1.1 Nguồn dữ liệu

Bộ dữ liệu **Body Fat** gồm số đo cơ thể của **252 nam giới** (tuổi từ 22 đến 81). Phần trăm mỡ cơ thể được ước lượng từ mật độ cơ thể đo bằng phương pháp cân dưới nước. Dữ liệu do Penrose, Nelson và Fisher (1985) thu thập và được Johnson (1996) giới thiệu trong *Journal of Statistics Education*, 4(1). Báo cáo dùng bản Kaggle của fedesoriano.

- Mô tả biến bản gốc: <https://jse.amstat.org/datasets/fat.txt>
- Bản Kaggle: <https://www.kaggle.com/datasets/fedesoriano/body-fat-prediction-dataset>
- Bản OpenML (id 560): <https://api.openml.org/d/560>; StatLib: <https://lib.stat.cmu.edu/datasets/bodyfat>

### 1.2 Các biến sử dụng

Trong 14 biến của bộ dữ liệu, báo cáo chỉ dùng 5 biến sau (đều là biến số liên tục, chưa dùng biến giả):

| Biến | Ý nghĩa | Đơn vị | Vai trò |
|---|---|---|---|
| `BodyFat` | Phần trăm mỡ cơ thể (công thức Siri) | % | **Y** |
| `Abdomen` | Vòng bụng | cm | x |
| `Wrist` | Vòng cổ tay | cm | x |
| `Age` | Tuổi | năm | x |
| `Height` | Chiều cao | inch | x (biến đối chứng) |

### 1.3 Lý do chọn từng biến x

- **`Abdomen`**: mỡ ở nam giới tích tụ nhiều ở vùng bụng, nên vòng bụng lớn thường đi kèm tỉ lệ mỡ cao.
- **`Age`**: tỉ lệ mỡ cơ thể có xu hướng tăng theo tuổi.
- **`Wrist`**: vòng cổ tay phản ánh kích thước khung xương; người có khung xương lớn thường nặng hơn nên có thể có tỉ lệ mỡ khác. Mối liên hệ này là gián tiếp.
- **`Height`**: chiều cao không có lý do rõ ràng ảnh hưởng đến tỉ lệ mỡ, được đưa vào làm **biến đối chứng** để xem một biến ít liên quan sẽ cho kết quả ra sao.

### 1.4 Các biến không dùng

- `Density`: `BodyFat` được tính trực tiếp từ biến này (`BodyFat = 495/Density − 450`), nên hồi quy theo `Density` là hiển nhiên và không có ý nghĩa phân tích.
- `Weight`, `Chest`, `Hip`, `Thigh`, ...: tương quan rất chặt với `Abdomen` (đa cộng tuyến), phù hợp để xét ở hồi quy bội hơn là hồi quy đơn. Số liệu tương quan sẽ dẫn ở mục 2.

### 1.5 Xử lý dữ liệu

Dữ liệu không thiếu và không trùng dòng. Chỉ **một ô** được sửa: quan sát số 42 có `Height` = 29,5 inch (khoảng 75 cm), rõ ràng là lỗi gõ nhầm vì người này nặng 205 lb và vòng bụng 104,3 cm; được sửa thành **69,5 inch** (Johnson, 1996 cũng đề xuất giá trị này). Các điểm đáng ngờ khác được **giữ nguyên** và sẽ nêu ở mục 4 (kiểm tra phần dư): case 182 (`BodyFat` = 0), case 48 và 76 (`BodyFat` lệch khỏi giá trị tính từ `Density`), case 39 (vòng bụng 148,1 cm, điểm đòn bẩy lớn).

### 1.6 Mục tiêu phân tích

Xét ảnh hưởng của **từng** số đo cơ thể (`Abdomen`, `Wrist`, `Age`, `Height`) đến `BodyFat` bằng hồi quy tuyến tính đơn: ước lượng hệ số, kiểm định ý nghĩa thống kê của hệ số góc, so sánh R² giữa các biến và kiểm tra các giả thiết của mô hình qua phần dư.

In [ ]:
# Đọc dữ liệu (file đã sửa case 42) và chọn các cột dùng trong báo cáo
df_full = pd.read_csv("bodyfat_cleaned.csv")
df = df_full[["BodyFat", "Abdomen", "Wrist", "Age", "Height"]].copy()

print("Kích thước toàn bộ dữ liệu:", df_full.shape)
print("Kích thước dữ liệu dùng để hồi quy:", df.shape)

# Kiểm tra ô đã sửa: case 42 là dòng chỉ số 41 (đếm từ 0)
print("\nCase 42 sau khi sửa:")
print(df_full.loc[41, ["Weight", "Height", "Abdomen", "BodyFat"]].to_string())

print("\n5 dòng đầu:")
display(df.head())

print("Thống kê mô tả:")
display(df.describe().round(2))

print("Số giá trị thiếu theo cột:")
print(df.isna().sum().to_string())

**Nhận xét.** Dữ liệu dùng để hồi quy gồm **n = 252** quan sát và 5 biến, không có giá trị thiếu. Chiều cao sau khi sửa case 42 nằm trong khoảng hợp lý (nhỏ nhất khoảng 64 inch). Các biến có đơn vị khác nhau (cm, inch, năm, %), vì vậy hệ số góc ở mục 3 sẽ được diễn giải theo đúng đơn vị của từng biến x và không so sánh trực tiếp độ lớn của các hệ số với nhau; việc so sánh giữa các biến dựa trên R².

## 2. Khảo sát sơ bộ

Trước khi ước lượng mô hình, ta xem hình dạng quan hệ giữa `BodyFat` và từng biến x bằng biểu đồ phân tán, rồi đo mức độ quan hệ tuyến tính bằng hệ số tương quan mẫu

$$r = \frac{S_{xy}}{\sqrt{S_{xx}\,SST}}.$$

Mức độ tương quan được xếp hạng theo bảng trong bài giảng (MI3031, mục 4.2.1): $|r|\in[0{,}11;0{,}25]$ yếu đến trung bình, $[0{,}26;0{,}50]$ trung bình đến mạnh, $[0{,}51;0{,}75]$ mạnh, $[0{,}76;0{,}85]$ rất mạnh, $>0{,}85$ hầu như hoàn hảo.

In [ ]:
xs = ["Abdomen", "Wrist", "Age", "Height"]
don_vi = {"Abdomen": "cm", "Wrist": "cm", "Age": "năm", "Height": "inch"}

# Hai điểm đáng chú ý (chỉ số đếm từ 0): case 39 (Abdomen = 148,1) và case 182 (BodyFat = 0)
dac_biet = {38: "case 39", 181: "case 182"}

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, x in zip(axes.ravel(), xs):
    r = df[x].corr(df["BodyFat"])
    ax.scatter(df[x], df["BodyFat"], s=14, alpha=0.6, edgecolor="none")
    b1, b0 = np.polyfit(df[x], df["BodyFat"], 1)          # đường thẳng bình phương nhỏ nhất (chỉ để minh họa)
    g = np.linspace(df[x].min(), df[x].max(), 100)
    ax.plot(g, b0 + b1 * g, color="crimson", lw=1.5)
    for i in dac_biet:                                      # khoanh tròn hai điểm đáng chú ý
        ax.scatter(df.loc[i, x], df.loc[i, "BodyFat"], s=70, facecolors="none", edgecolors="black", lw=1.2)
    ax.set_xlabel(f"{x} ({don_vi[x]})")
    ax.set_ylabel("BodyFat (%)")
    ax.set_title(f"BodyFat theo {x}: r = {r:.3f}")
fig.suptitle("Biểu đồ phân tán (vòng tròn đen: case 39 và case 182)", y=1.0)
plt.tight_layout()
plt.show()

In [ ]:
# Ma trận hệ số tương quan mẫu của Y và 4 biến x
corr = df.corr()
print("Ma trận tương quan:")
display(corr.round(3))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9,
                color="white" if abs(v) > 0.6 else "black")
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Ma trận tương quan")
plt.tight_layout()
plt.show()

In [ ]:
# Xếp hạng mức tương quan giữa Y và từng x theo bảng của bài giảng,
# kèm kiểm định H0: rho = 0 (t0 = r*sqrt(n-2)/sqrt(1-r^2) ~ t(n-2) nếu H0 đúng)
def xep_hang(r):
    a = abs(r)
    if a == 0:    return "Không tương quan"
    if a <= 0.10: return "Rất yếu"
    if a <= 0.25: return "Yếu đến trung bình"
    if a <= 0.50: return "Trung bình đến mạnh"
    if a <= 0.75: return "Mạnh"
    if a <= 0.85: return "Rất mạnh"
    return "Hầu như hoàn hảo"

n = len(df)
dong = []
for x in xs:
    r, p = stats.pearsonr(df[x], df["BodyFat"])
    t0 = r * np.sqrt(n - 2) / np.sqrt(1 - r**2)
    dong.append({"x": x, "r": round(r, 3), "r²": round(r**2, 3),
                 "Chiều": "thuận" if r > 0 else "nghịch",
                 "Xếp hạng |r|": xep_hang(r), "t0": round(t0, 2), "p-value": f"{p:.2e}"})
display(pd.DataFrame(dong).set_index("x"))

# Các biến không đưa vào hồi quy: tương quan với Abdomen và với BodyFat
bo = ["Weight", "Chest", "Hip", "Thigh"]
display(pd.DataFrame({"r với Abdomen": df_full[bo].corrwith(df_full["Abdomen"]),
                      "r với BodyFat": df_full[bo].corrwith(df_full["BodyFat"])}).round(3))

**Nhận xét về dạng, chiều và mức độ quan hệ** (theo biểu đồ phân tán và bảng xếp hạng ở trên):

- **`Abdomen`**: quan hệ **thuận**, các điểm tập trung quanh một đường thẳng; $r = 0{,}813$ thuộc nhóm **rất mạnh** và vượt ngưỡng $0{,}76$ mà bài giảng dùng để coi quan hệ là xấp xỉ tuyến tính. Độ phân tán của `BodyFat` quanh đường thẳng khá đều ở các mức vòng bụng khác nhau, chưa thấy dạng phễu. Hai điểm tách khỏi đám mây là case 39 (vòng bụng 148,1 cm, `BodyFat` chỉ 35,2%, nằm thấp hơn nhiều so với đường thẳng) và case 182 (`BodyFat` = 0).
- **`Wrist`**: quan hệ **thuận nhưng yếu về mặt dự đoán**; $r = 0{,}347$ thuộc nhóm **trung bình đến mạnh**. Các điểm phân tán rộng quanh đường thẳng.
- **`Age`**: quan hệ **thuận**, $r = 0{,}291$ thuộc nhóm **trung bình đến mạnh**, đám mây điểm cũng rộng.
- **`Height`**: đường thẳng gần như nằm ngang, $r = -0{,}025$ thuộc nhóm **rất yếu**. Kiểm định $H_0:\rho=0$ cho $p \approx 0{,}69$, nên không có bằng chứng về tương quan tuyến tính giữa chiều cao và `BodyFat`, đúng với vai trò biến đối chứng.

Vì $R^2 = r^2$ trong hồi quy tuyến tính đơn, ta dự đoán trước `Abdomen` sẽ giải thích khoảng 66% biến thiên của `BodyFat`, `Wrist` khoảng 12%, `Age` khoảng 8,5% và `Height` gần như không giải thích được gì. Mục 3 sẽ kiểm chứng lại.

**Tương quan giữa các biến x.** Cặp cao nhất là `Abdomen`–`Wrist` ($r = 0{,}62$, mạnh), cho thấy `Wrist` có thể "mượn" một phần thông tin từ `Abdomen`; tiếp theo là `Wrist`–`Height` ($0{,}40$). Các cặp còn lại đều có $|r| \le 0{,}25$. Vì mỗi mô hình ở mục 3 chỉ có một biến x nên điều này chưa gây vấn đề, nhưng cần lưu ý khi mở rộng sang hồi quy bội.

**Chọn biến để hồi quy.** Giữ cả bốn biến `Abdomen`, `Wrist`, `Age`, `Height` cho mục 3: ba biến đầu có tương quan thuận có ý nghĩa với `BodyFat` ($p < 10^{-5}$), còn `Height` được giữ có chủ ý làm trường hợp đối chứng không có quan hệ. Các biến `Weight`, `Chest`, `Hip`, `Thigh` được bỏ vì tương quan với `Abdomen` rất cao ($0{,}77$ đến $0{,}92$), tức chúng gần như đo cùng một thông tin với `Abdomen` và chỉ nên xét cùng nhau trong hồi quy bội; riêng `Density` bị bỏ vì `BodyFat` được tính trực tiếp từ nó.